In [4]:
import os
import uuid
import random
import json
import sys
import time
import re
import ast
from enum import Enum
from typing import Optional, List, Dict, Any
from dataclasses import dataclass, field
from datetime import datetime, timezone
from tqdm import tqdm
import pandas as pd
import numpy as np
import requests
from functools import partial
from requests.auth import HTTPBasicAuth
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio
from tqdm.asyncio import tqdm_asyncio
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openai import AsyncOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langfuse import Langfuse
from dotenv import load_dotenv
load_dotenv()


from pathlib import Path
# Корень проекта Executive_Exocortex
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv()

# atomizer, linker, GraphRAG
from config.settings import settings
from zettelkasten.atomizer import NoteAtomizer
from zettelkasten.linker import (
    GraphLinker,
    LocalEmbeddingModel,
    LinkAction,
    LinkDecision,
    LinkResult,
)

from zettelkasten.atomizer import ZettelCard
from storage.neo4j.client import get_neo4j_client
from storage.neo4j.schema import init_schema
from storage.neo4j.repository import ZettelRepository

/Users/nastya/github/Executive_Exocortex/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# инициализация компонентов пайплайна
embedding_model = LocalEmbeddingModel(
    model_name=settings.embedding_model_name,  # multilanguafe-e5-base
)

count_syntetic_data = 200  # кол-во сгенерированных синтетических данных (200 заметок, которые подробятся на отдельные мысли)
generate_llm_model = 'openai/gpt-5.1'  # модель которая будет генерировать синтетические данные
llm_models = [
    'openai/gpt-4o-mini',
    'google/gemini-2.5-flash',
    'anthropic/claude-haiku-4.5',
]

2026-06-28 16:19:55,097 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-28 16:19:55,371 | INFO | Loading SentenceTransformer model from intfloat/multilingual-e5-base.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11418.46it/s]


[EmbeddingModel] intfloat/multilingual-e5-base загружена за 8.2с (dim=768, device=mps)


/Users/nastya/github/Executive_Exocortex/zettelkasten/linker.py:84: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


# Synthetic notes  from top-managers

In [3]:
# Алгоритм

# 1. сильеная модель генерирует заметки (синтетические данные)
# - кол-во предложений в заметке определяется рандомно (3-10 предложений)
# 2. тест атомайзера
# - Сильная модель разбивает эти заметки на атомарные мысли (zettel-карточки) по реальному скрипту atomizer (с тем же промптом)
# - получаем эталонный список мыслей по заметкам
# - получаем эталонный граф связей по заметкам
# - Импортировать этот набор в Langfuse Dataset.
# - тестируем более слабые модели на атомайзере
# 
# 3. тест линкера   
# 
# - тестируем более слабые модели на атомайзере
# - на вход подаем заметку
# - на выходе получаем список zettel-карточек
# - сравниваем с эталонным списком
# - считаем метрики
# - выводим результаты
# 4. выбираем подходящую модель (соотношение качество/цена) и на ней перебирваем системный промпт

# Автоматически записывать в Langfuse:
# стоимость;
# задержку;
# число входных и выходных токенов;
# пользовательские метрики (полнота, точность, корректность JSON, число ошибок и т.д.).


In [4]:
class SyntheticNote(BaseModel):
    """Синтетическая заметка для бенчмарка"""
    note_id: str = Field(default_factory=lambda: str(uuid.uuid4())) # уникальный идентификатор заметки
    note_text: str = Field(description="Текст заметки") # текст заметки         
    num_sentences: int = Field(description="Количество предложений в заметке") # количество предложений в заметке
    domain: str = Field(description="Предметная область") # предметная область
    style: str = Field(description="Стиль заметки") # стиль заметки
    metadata: dict = Field(default_factory=dict) # метаданные заметки
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc)) # дата создания заметки


class SyntheticNotesDataset(BaseModel):
    """Датасет синтетических заметок"""
    dataset_id: str = Field(default_factory=lambda: str(uuid.uuid4())) # уникальный идентификатор датасета
    notes: List[SyntheticNote] = Field(description="Список заметок") # список заметок
    generation_params: dict = Field(default_factory=dict) # параметры генерации
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc)) # дата создания датасета


In [5]:
# домены и стили заметок    
BUSINESS_DOMAINS = [
    "продуктовая разработка",
    "продажи и маркетинг", 
    "управление персоналом",
    "финансы и инвестиции",
    "операционная деятельность",
    "стратегия и развитие",
    "работа с клиентами",
    "аналитика и метрики",
    'спортивный проект',
    'культурный проект',
    'образовательный проект',
    'благотворительный проект',
    'реструктуризация компании',
    'изменение стратегии разваития проекта',
    'обсуждение перспектив развития компании',
    'ИИ-зация процессов'
]

NOTE_STYLES = [
    "расшифровка встречи",
    "голосовая заметка",
    "рабочие мысли",
    "записи после звонка",
    "планерка/совещание",
    "анализ ситуации",
    "список задач и решений",
]



In [6]:

SYNTHETIC_NOTE_GENERATION_SYSTEM_PROMPT = """
Ты — генератор реалистичных бизнес-заметок топ-менеджеров для тестирования систем обработки знаний.

Твоя задача: создать естественную заметку, которая выглядит как реальная запись топ-менеджера.

ТРЕБОВАНИЯ К ЗАМЕТКАМ:

1. РЕАЛИСТИЧНОСТЬ:
   - Пиши как реальный человек записывает мысли после встречи, звонка или во время размышлений
   - Используй естественный деловой язык, не слишком формальный
   - Допустимы разговорные обороты, характерные для устной речи
   - Могут быть сокращения, аббревиатуры, принятые в бизнесе
   - не соблюдай пунктуацию, не используй кавычки, больше реалистичности в текст
   - текст может быть похож как будто он записан быстро под диктовку на скорую руку, общими  тезисами
   
2. СТРУКТУРА:
   - Заметка должна содержать ТОЧНО указанное количество предложений
   - Предложения могут быть разной длины (от коротких до развернутых)
   - Логика повествования: последовательная, но может включать переходы между темами
   - Допустимы вводные фразы типа "Встреча с...", "Обсудили...", "Важно..."

3. СОДЕРЖАНИЕ (смешивай разные типы информации):
   - ФАКТЫ: конкретные данные, метрики, результаты ("Выручка упала на 15%", "В команде 8 человек")
   - РЕШЕНИЯ: что решили делать ("Закрываем проект X", "Переходим на новую CRM")
   - ЗАДАЧИ: поручения с исполнителями или без ("Петров готовит презентацию к среде")
   - РИСКИ: что может пойти не так ("Поставщик может сорвать дедлайн")
   - ИДЕИ: предложения, гипотезы ("Попробовать A/B тест новой главной")
   - ВОПРОСЫ: что нужно выяснить ("Почему конверсия упала?")
   - КОНТЕКСТ: фоновая информация ("Клиент недоволен последним релизом")

4. СУЩНОСТИ (обязательно включай):
   - Имена людей: 
   - Проекты: кодовые названия
   - Компании: клиенты, партнеры, конкуренты 
   - Технологии: конкретные инструменты 
   - Метрики и цифры: проценты, суммы, даты, дедлайны

5. РАЗНООБРАЗИЕ ТЕМ (варьируй):
   - Продуктовая разработка (фичи, релизы, баги, техдолг и другие)
   - Продажи и маркетинг (лиды, конверсия, CAC, LTV, кампании и другие)
   - HR (найм, увольнения, обучение, мотивация и другие)
   - Финансы (бюджеты, расходы, инвестиции, прибыль и другие )
   - Операционка (процессы, поставщики, логистика и другие)
   - Стратегия (цели, конкуренты, рынки, партнерства и другие)
   - Другое (любая тема, с которой может столкнуться топ-менеджер и другие)

6. СТИЛИ ЗАМЕТОК (варьируй):
   - Расшифровка встречи: "Встреча с командой продукта. Обсудили..."
   - Голосовая заметка: "Только что созвонился с Петровым. Говорит что..."
   - Рабочие мысли: "Подумал про запуск рекламы. Нужно..."
   - Совещание: "Планерка отдела продаж. Иванов отчитался..."
   - Записи после звонка: "Звонок с клиентом Х. Они хотят..."
   - Побуждение к действию/напоминания: "Нужно срочно разобраться с ..."

7. СВЯЗНОСТЬ:
   - Мысли в заметке должны быть хотя бы минимально связаны общим контекстом
   - Допустимы переходы между темами ("Еще обсудили...", "Отдельно...")
   - Можно развивать одну тему через несколько предложений
   - Можно затрагивать разные темы в рамках одного контекста (встреча, день)

8. МЕСТОИМЕНИЯ И ССЫЛКИ:
   - ИСПОЛЬЗУЙ местоимения и контекстные ссылки ("он", "она", "это", "там")
   - Это естественно для заметок и создаст реалистичную задачу для разюиения текста на отдельные мысли
   - Пример: "Встретился с Петровым. Он сказал, что проект задерживается. Это критично."

ПРИМЕРЫ ХОРОШИХ ЗАМЕТОК:

Пример 1 (4 предложения, встреча):
"Встреча с Ивановым по проекту Альфа. Он сказал что запуск переносится на конец квартала из-за проблем с API. Нужно обсудить с командой план Б. Также он предложил использовать готовое решение от Amazon вместо разработки своего."

Пример 2 (6 предложений, метрики):
"Смотрел дашборд по продажам за март. Выручка упала на 12% по сравнению с февралем. CAC вырос до 5000 рублей, это плохо. Нужно срочно разобраться с эффективностью рекламных каналов. Сидоров обещал сделать анализ к пятнице. Возможно стоит приостановить Яндекс Директ до выяснения."

Пример 3 (5 предложений, HR):
"Интервью с кандидатом на позицию senior backend. Сильный спец, опыт в Яндексе 4 года, знает Go и Python. Хочет 350к, это выше бюджета на 50к. Но команде нужно усиление срочно, проект Mars буксует. Согласовать с финансами повышение бюджета."

Пример 4 (7 предложений, стратегия):
"Думал про выход на рынок Казахстана. Конкуренты там слабые, можем быстро захватить долю. Но нужна локальная команда продаж, минимум 3-5 человек. Это примерно 2 млн рублей в месяц на ФОТ. Окупится ли за полгода? Нужно попросить аналитиков посчитать юнит-экономику. Также узнать про местное законодательство и налоги."

ВАЖНО:
- Генерируй ТОЛЬКО текст заметки
- НЕ добавляй заголовки, метаданные, пояснения
- НЕ используй markdown или форматирование
- Просто естественный текст заметки
- Количество предложений должно быть ТОЧНО как указано в запросе
"""


SYNTHETIC_NOTE_GENERATION_USER_PROMPT_TEMPLATE = """
Сгенерируй реалистичную бизнес-заметку топ-менеджера.

Параметры:
- Количество предложений: {num_sentences}
- Предметная область: {domain}
- Стиль: {style}

Верни только текст заметки, без дополнительных пояснений.
"""

In [8]:
class SyntheticNoteGenerator:
    """Генератор синтетических заметок для бенчмарка"""
    
    def __init__(
        self,
        model_name: str = "openai/gpt-4o",
        temperature: float = 0.95,  # высокая температура для разнообразия
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
    ):
        self.model_name = model_name
        self.temperature = temperature
        
        self.llm = ChatOpenAI(
            model=self.model_name,
            api_key=api_key or os.getenv("LLM_API_KEY"),
            base_url=base_url or os.getenv("LLM_BASE_URL"),
            temperature=self.temperature,
        )
        
        # Для потокобезопасной работы с Excel
        self.excel_lock = Lock()
    
    def generate_note(
        self,
        num_sentences: Optional[int] = None,
        domain: Optional[str] = None,
        style: Optional[str] = None,
    ) -> SyntheticNote:
        """
        Генерирует одну синтетическую заметку.
        
        Args:
            num_sentences: количество предложений ( выбирается случайно 3-10)
            domain: предметная область ( выбирается случайно)
            style: стиль заметки ( выбирается случайно)
        
        Returns:
            SyntheticNote
        """
        # Определяем параметры, случайно определяем количество предложений, предметную область и стиль заметки
        if num_sentences is None:
            num_sentences = random.randint(3, 10)
        
        if domain is None:
            domain = random.choice(BUSINESS_DOMAINS)
        
        if style is None:
            style = random.choice(NOTE_STYLES)
        
        # Генерируем заметку через LLM
        messages = [
            SystemMessage(content=SYNTHETIC_NOTE_GENERATION_SYSTEM_PROMPT),
            HumanMessage(content=SYNTHETIC_NOTE_GENERATION_USER_PROMPT_TEMPLATE.format(
                num_sentences=num_sentences,
                domain=domain,
                style=style,
            )),
        ]
        
        response = self.llm.invoke(messages)
        note_text = response.content.strip()
        
        # Валидируем количество предложений (примерно)
        actual_sentences = self._count_sentences(note_text)
        
        return SyntheticNote(
            note_text=note_text,
            num_sentences=num_sentences,
            domain=domain,
            style=style,
            metadata={
                "actual_sentences_count": actual_sentences,
                "model_name": self.model_name,
                "temperature": self.temperature,
            }
        )
    
    def _generate_note_wrapper(self, params: dict) -> Optional[SyntheticNote]:
        """
        Обертка для генерации заметки с обработкой ошибок.
        Используется для параллельной генерации.
        """
        try:
            return self.generate_note(
                num_sentences=params['num_sentences'],
                domain=params['domain'],
                style=params['style']
            )
        except Exception as e:
            print(f"\n✗ Ошибка при генерации заметки: {e}")
            return None
    
    def generate_dataset(
        self,
        num_notes: int = 100,
        min_sentences: int = 3,
        max_sentences: int = 10,
        domains: Optional[List[str]] = None,
        styles: Optional[List[str]] = None,
        sentence_distribution: Optional[dict] = None,
        output_excel_path: Optional[str] = None,
        save_every: int = 10,
        max_workers: int = 5,
    ) -> SyntheticNotesDataset:
        """
        Генерирует датасет синтетических заметок с параллельной обработкой.
        
        Args:
            num_notes: количество заметок
            min_sentences: минимальное количество предложений
            max_sentences: максимальное количество предложений
            domains: список доменов (None = использовать все)
            styles: список стилей (None = использовать все)
            sentence_distribution: распределение по кол-ву предложений
                Пример: {3: 0.2, 5: 0.3, 7: 0.3, 10: 0.2}
            output_excel_path: путь для сохранения Excel файла
            save_every: сохранять каждые N заметок
            max_workers: количество параллельных потоков
        
        Returns:
            SyntheticNotesDataset
        """
        if domains is None:
            domains = BUSINESS_DOMAINS
        
        if styles is None:
            styles = NOTE_STYLES
        
        notes = []
        
        # Подготовка распределения
        if sentence_distribution:
            sentence_pool = []
            for num_sent, weight in sentence_distribution.items():
                count = int(num_notes * weight)
                sentence_pool.extend([num_sent] * count)
            # Дополняем до num_notes если нужно
            while len(sentence_pool) < num_notes:
                sentence_pool.append(random.choice(list(sentence_distribution.keys())))
            random.shuffle(sentence_pool)
        else:
            sentence_pool = [random.randint(min_sentences, max_sentences) for _ in range(num_notes)]
        
        print(f"Генерация {num_notes} синтетических заметок...")
        
        # Подготовка параметров для каждой заметки
        generation_params_list = []
        for i in range(num_notes):
            #  случайно выбираем количество предложений, предметную область и стиль заметки
            num_sentences = sentence_pool[i]
            domain = random.choice(domains)
            style = random.choice(styles)
            
            generation_params_list.append({
                'num_sentences': num_sentences,
                'domain': domain,
                'style': style,
                'index': i
            })
        
        # Параллельная генерация заметок
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Запускаем все задачи
            future_to_params = {
                executor.submit(self._generate_note_wrapper, params): params 
                for params in generation_params_list
            }
            
            # Обрабатываем результаты по мере готовности
            with tqdm(total=num_notes, desc="Генерация заметок") as pbar:
                for future in as_completed(future_to_params):
                    params = future_to_params[future]
                    # генерируем заметку
                    try:
                        note = future.result()
                        if note:
                            notes.append(note)
                            
                            # Сохраняем каждые N заметок
                            if output_excel_path and len(notes) % save_every == 0:
                                self._save_notes_to_excel(notes, output_excel_path)
                                tqdm.write(f"✓ Сохранено {len(notes)} заметок в Excel")
                        
                    except Exception as e:
                        tqdm.write(f"✗ Ошибка при обработке результата: {e}")
                    
                    pbar.update(1)
        
        # Финальное сохранение
        if output_excel_path and notes:
            self._save_notes_to_excel(notes, output_excel_path)
            print(f"\n✓ Сохранено {len(notes)} заметок в {output_excel_path}")
        
        # Статистика
        sentence_stats = {}
        for note in notes:
            count = note.num_sentences
            sentence_stats[count] = sentence_stats.get(count, 0) + 1
        
        dataset = SyntheticNotesDataset(
            notes=notes,
            generation_params={
                "num_notes_requested": num_notes,
                "num_notes_generated": len(notes),
                "min_sentences": min_sentences,
                "max_sentences": max_sentences,
                "domains": domains,
                "styles": styles,
                "model_name": self.model_name,
                "temperature": self.temperature,
                "sentence_distribution": sentence_stats,
                "max_workers": max_workers,
            }
        )
        
        return dataset
    
    def _count_sentences(self, text: str) -> int:
        """Подсчитывает количество предложений в тексте (примерно)"""
        import re
        # Простая эвристика: считаем по точкам, вопросам, восклицаниям
        sentences = re.split(r'[.!?]+', text)
        return len([s for s in sentences if s.strip()])
    
    def _save_notes_to_excel(self, notes: List[SyntheticNote], output_path: str):
        """
        Сохраняет заметки в Excel файл (потокобезопасно).
        
        Args:
            notes: список заметок
            output_path: путь к Excel файлу
        """
        with self.excel_lock:
            # Создаем DataFrame
            data = []
            for i, note in enumerate(notes, 1):
                data.append({
                    '№': i,
                    'note_id': note.note_id,
                    'note_text': note.note_text,
                    'num_sentences': note.num_sentences,
                    'actual_sentences': note.metadata.get('actual_sentences_count', 0),
                    'domain': note.domain,
                    'style': note.style,
                    'created_at': note.created_at.strftime('%Y-%m-%d %H:%M:%S'),
                })
            
            df = pd.DataFrame(data)
            
            # Создаем директорию если не существует
            output_file = Path(output_path)
            output_file.parent.mkdir(parents=True, exist_ok=True)
            
            # Сохраняем в Excel
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                df.to_excel(writer, sheet_name='Заметки', index=False)
                
                # Автоматическая ширина колонок
                worksheet = writer.sheets['Заметки']
                for idx, col in enumerate(df.columns):
                    max_length = max(
                        df[col].astype(str).apply(len).max(),
                        len(str(col))
                    )
                    # Ограничиваем максимальную ширину
                    max_length = min(max_length, 100)
                    worksheet.column_dimensions[chr(65 + idx)].width = max_length + 2
    
    def save_dataset(
        self,
        dataset: SyntheticNotesDataset,
        output_path: str,
        save_pretty: bool = True,
    ):
        """
        Сохраняет датасет в JSON файл.
        
        Args:
            dataset: датасет для сохранения
            output_path: путь к выходному файлу
            save_pretty: сохранять с отступами для читаемости
        """
        output_file = Path(output_path)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_file, 'w', encoding='utf-8') as f:
            if save_pretty:
                json.dump(
                    dataset.model_dump(mode='json'),
                    f,
                    ensure_ascii=False,
                    indent=2
                )
            else:
                json.dump(
                    dataset.model_dump(mode='json'),
                    f,
                    ensure_ascii=False
                )

In [ ]:
def generate_notes_dataset():
    """Пример генерации датасета"""
    
    # Создаем генератор
    generator = SyntheticNoteGenerator(
        model_name=generate_llm_model,  # используем сильную модель для генерации
        temperature=0.95,
    )
    
    # Генерируем датасет с параллельной обработкой
    dataset = generator.generate_dataset(
        num_notes=count_syntetic_data,
        min_sentences=3,
        max_sentences=10,
        output_excel_path="synthetic_datasets/notes_dataset_v1.xlsx",
        save_every=50,  # сохранять каждые 50 заметок
        max_workers=10,  # 10 параллельных потоков
    )


In [10]:
generate_notes_dataset()

Генерация 200 синтетических заметок...


Генерация заметок:  25%|██▌       | 50/200 [01:16<03:43,  1.49s/it]

✓ Сохранено 50 заметок в Excel


Генерация заметок:  50%|█████     | 100/200 [02:28<03:08,  1.88s/it]

✓ Сохранено 100 заметок в Excel


Генерация заметок:  75%|███████▌  | 150/200 [03:32<01:11,  1.43s/it]

✓ Сохранено 150 заметок в Excel


Генерация заметок: 100%|██████████| 200/200 [04:51<00:00,  1.46s/it]

✓ Сохранено 200 заметок в Excel

✓ Сохранено 200 заметок в eval/synthetic_datasets/notes_dataset_v1.xlsx


In [27]:
# Atomixzer

# Atomizer

## reference thoughts

In [7]:
reference_llm_model = 'openai/gpt-5.2'

df_notes = pd.read_excel("synthetic_datasets/notes_dataset_v1.xlsx")
df_notes.head(3)

,№,note_id,note_text,num_sentences,actual_sentences,domain,style,created_at
0,1,c09941ef-4576-4710-936d-dc12911f0ab7,Думаю по реструктуризации после разговора с Ма...,4,4,реструктуризация компании,рабочие мысли,2026-06-27 11:49:56
1,2,1d4f8aa3-9dc0-4898-b324-3b3c61a67436,Разобрал цифры по благотворительному проекту С...,3,3,благотворительный проект,анализ ситуации,2026-06-27 11:49:56
2,3,3a046e1a-56fb-4abb-afc4-cb65d85d3a24,Звонок с клиентом Северсталь Диджитал по проек...,6,6,работа с клиентами,записи после звонка,2026-06-27 11:49:58


In [8]:
# прогнояем заметки через реальный пайплайн с сильной моделью и считаем ее ответ за эталонные

atomizer = NoteAtomizer(
    model_name=reference_llm_model,  # сильная модель, берем как эталон
    temperature=settings.zettel_atomizer_temperature,
    system_prompt=settings.zettel_atomizer_system_prompt,
    user_prompt_template=settings.zettel_atomizer_user_prompt_template,
)

In [9]:
def process_single_note(args):
    """
    Функция обработки одной заметки, которая будет выполняться в отдельном потоке.
    args: кортеж (index, note_id, note_text)
    """
    idx, note_id, note_text = args
    
    try:
        # Передаем порядковый индекс вместо инкрементируемого счетчика
        raw_cards = atomizer.atomize(
            text=note_text, 
            current_db_max_root_id=idx
        )
        return note_id, note_text, raw_cards
    except Exception as e:
        # Ловим критические ошибки самого потока, если они возникнут
        return note_id, note_text, f"ошибка потока: {e}"


In [10]:
MAX_WORKERS = 15 

df_notes_clean = df_notes.reset_index(drop=True)
tasks = [
    (idx, row['note_id'], row['note_text']) 
    for idx, row in df_notes_clean.iterrows()
]

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # executor.map сохраняет исходный порядок элементов, что очень удобно
    # Оборачиваем в list(tqdm(...)) для отображения прогресс-бара
    results = list(tqdm(
        executor.map(process_single_note, tasks), 
        total=len(tasks), 
        desc="Генерация мыслей через LLM"
    ))

df_thoughts_list = []
for note_id, note_text, raw_cards in results:
    # Проверка: ваш атомайзер в случае ошибки возвращает строку, а не список объектов ZettelCard
    if isinstance(raw_cards, str):
        print(f"[ERROR] Ошибка обработки заметки {note_id}: {raw_cards}")
        continue
        
    if not raw_cards:
        continue

    # Разворачиваем список карточек в строки датафрейма
    for card in raw_cards:
        df_thoughts_list.append({
            'note_id': note_id,
            'note_text': note_text,
            'zettel_id': card.zettel_id,
            'luhmann_id': card.luhmann_id,
            'parent_id': card.parent_id,
            'parent_luhmann_id': card.parent_luhmann_id,
            'content': card.content,
            'thought_type': card.thought_type,
            # Переводим списки в строку, так как Excel не умеет хранить нативные list структуры в ячейке
            'tags': ", ".join(card.tags) if card.tags else "",
            'parent_hint': card.parent_hint,
            'is_root_topic': card.is_root_topic,
            'created_at': card.created_at,
            # Эмбеддинги также лучше перевести в строку или исключить, если они не нужны в самом Excel
            'embedding': str(card.embedding) if card.embedding else None
        })

# Сохранение в Excel
if df_thoughts_list:
    df_thoughts = pd.DataFrame(df_thoughts_list)
    
    
    output_path = "synthetic_datasets/reference_thoughts_v1.xlsx"
    df_thoughts.to_excel(output_path, index=False)
    print(f"\Файл сохранен {output_path}. Всего сгенерировано мыслей: {len(df_thoughts)}")


Генерация мыслей через LLM: 100%|██████████| 200/200 [04:08<00:00,  1.24s/it]


\Файл сохранен synthetic_datasets/reference_thoughts_v1.xlsx. Всего сгенерировано мыслей: 3295


In [11]:
df_thoughts.shape

(3295, 13)

In [12]:
# считаем сколько мыслей в среднем генерируется на заметку
df_thoughts.groupby('note_id')['zettel_id'].nunique().mean()

np.float64(16.475)

In [13]:
# считаем сколько мыслей в среднем генерируется на заметку в зависимости от количества предложений в заметке
cnt_thoungts_by_sentences = df_thoughts.merge(
    df_notes,
    on='note_id',
    how='outer'  
).groupby(['note_id', 'num_sentences'])['zettel_id'].nunique().reset_index().groupby('num_sentences')['zettel_id'].mean().reset_index()
cnt_thoungts_by_sentences

,num_sentences,zettel_id
0,3,10.230769
1,4,12.380952
2,5,12.526316
3,6,15.771429
4,7,18.050000
5,8,19.357143
6,9,20.478261
7,10,21.607143


In [14]:
public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
secret_key = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = "http://localhost:3000"

DATASET_NAME = "atomizer-benchmark-v1"

# Читаем данные
df = pd.read_excel("synthetic_datasets/reference_thoughts_v1.xlsx")
grouped = df.groupby('note_id')

print("Загрузка данных в Langfuse через REST API...")

# Настраиваем авторизацию и URL для API
auth = HTTPBasicAuth(public_key, secret_key)
api_url = f"{LANGFUSE_HOST}/api/public/dataset-items"

for note_id, group in tqdm(grouped, desc="Отправка заметок"):
    note_text = str(group.iloc[0]['note_text'])
    
    expected_cards = []
    for _, row in group.iterrows():
        expected_cards.append({
            "content": str(row['content']),
            "thought_type": str(row['thought_type']),
            "tags": str(row['tags'])
        })
    
    # Формируем JSON строго по REST API спецификации Langfuse
    payload = {
        "datasetName": DATASET_NAME,
        "input": {"note_text": note_text},
        "expectedOutput": {"cards": expected_cards}
    }
    
    # Отправляем прямой запрос
    response = requests.post(api_url, json=payload, auth=auth)
    
    if response.status_code not in (200, 201):
        tqdm.write(f"Ошибка отправки заметки {note_id}: {response.text}")

print(f"Датасет '{DATASET_NAME}'  загружен в Langfuse!")

Загрузка данных в Langfuse через REST API...


Отправка заметок: 100%|██████████| 200/200 [00:02<00:00, 89.80it/s] 

Датасет 'atomizer-benchmark-v1'  загружен в Langfuse!


## test models

Прогоняем на всех тестовых моделях (более слабых)

In [ ]:
llm_models = [
    'openai/gpt-4o-mini',
    'google/gemini-2.5-flash',
    'anthropic/claude-haiku-4.5',
]
MAX_WORKERS = 15 

In [16]:
def process_single_note(task, atomizer):
    """Обрабатывает одну заметку и возвращает результат + latency"""
    idx, note_id, note_text = task
    
    start_time = time.perf_counter()  # старт замера
    
    try:
        raw_cards = atomizer.atomize(note_text)
    except Exception as e:
        print(f"Ошибка при обработке {note_id}: {e}")
        raw_cards = None
    
    end_time = time.perf_counter()  # конец замера
    latency_ms = (end_time - start_time) * 1000  # в миллисекундах
    
    return note_id, note_text, raw_cards, latency_ms  # добавили latency



In [17]:
df_notes_clean = df_notes.reset_index(drop=True)
tasks = [
    (idx, row['note_id'], row['note_text']) 
    for idx, row in df_notes_clean.iterrows()
]

for model in llm_models:
    print(f"\nLLM model: {model}\n{'='*40}")
    
    atomizer = NoteAtomizer(
        model_name=model,
        temperature=settings.zettel_atomizer_temperature,
        system_prompt=settings.zettel_atomizer_system_prompt,
        user_prompt_template=settings.zettel_atomizer_user_prompt_template,
    )
    
    worker_func = partial(process_single_note, atomizer=atomizer)
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(
            executor.map(worker_func, tasks),
            total=len(tasks),
            desc=f"Обработка {model}"
        ))
    
    df_thoughts_list = []
    
    for note_id, note_text, raw_cards, latency_ms in results:  # распаковываем latency
        if isinstance(raw_cards, str) or not raw_cards:
            continue
        
        # latency на одну карточку — делим на количество карточек
        # Общая latency записывается в каждую строку для агрегации
        cards_count = len(raw_cards)
        latency_per_card = latency_ms / cards_count if cards_count > 0 else latency_ms
        
        for card in raw_cards:
            df_thoughts_list.append({
                'model_name': model,
                'note_id': note_id,
                'note_text': note_text,
                'zettel_id': card.zettel_id,
                'luhmann_id': card.luhmann_id,
                'parent_id': card.parent_id,
                'parent_luhmann_id': card.parent_luhmann_id,
                'content': card.content,
                'thought_type': card.thought_type,
                'tags': ", ".join(card.tags) if card.tags else "",
                'parent_hint': card.parent_hint,
                'is_root_topic': card.is_root_topic,
                'created_at': card.created_at,
                'embedding': str(card.embedding) if card.embedding else None,
                # ── новые колонки ──
                'latency_ms_total': round(latency_ms, 2),      # общее время на заметку
                'latency_ms_per_card': round(latency_per_card, 2),  # среднее на карточку
                'cards_count': cards_count,                     # сколько карточек вышло
            })

    
    if df_thoughts_list:
        df_model = pd.DataFrame(df_thoughts_list)
        safe_model_name = model.replace("/", "_").replace(":", "_")
        output_path = f"synthetic_datasets/thoughts_{safe_model_name}.xlsx"
        df_model.to_excel(output_path, index=False)
        
        # Печатаем сводку по латентности
        print(f"\nLatency summary для {model}:")
        print(f"  Среднее время на заметку: {df_model['latency_ms_total'].mean():.0f} мс")
        print(f"  Медиана:                  {df_model['latency_ms_total'].median():.0f} мс")
        print(f"  P95:                      {df_model['latency_ms_total'].quantile(0.95):.0f} мс")
        print(f"  Среднее карточек/заметку: {df_model['cards_count'].mean():.1f}")
        print(f"  Результаты сохранены в {output_path}")


LLM model: openai/gpt-4o-mini


Обработка openai/gpt-4o-mini: 100%|██████████| 200/200 [02:10<00:00,  1.53it/s]



Latency summary для openai/gpt-4o-mini:
  Среднее время на заметку: 10005 мс
  Медиана:                  9604 мс
  P95:                      15310 мс
  Среднее карточек/заметку: 12.9
  Результаты сохранены в synthetic_datasets/thoughts_openai_gpt-4o-mini.xlsx

LLM model: google/gemini-2.5-flash


Обработка google/gemini-2.5-flash: 100%|██████████| 200/200 [01:31<00:00,  2.19it/s]



Latency summary для google/gemini-2.5-flash:
  Среднее время на заметку: 6910 мс
  Медиана:                  6816 мс
  P95:                      10234 мс
  Среднее карточек/заметку: 18.0
  Результаты сохранены в synthetic_datasets/thoughts_google_gemini-2.5-flash.xlsx

LLM model: anthropic/claude-haiku-4.5


Обработка anthropic/claude-haiku-4.5: 100%|██████████| 200/200 [01:36<00:00,  2.07it/s]



Latency summary для anthropic/claude-haiku-4.5:
  Среднее время на заметку: 7200 мс
  Медиана:                  7235 мс
  P95:                      9962 мс
  Среднее карточек/заметку: 12.3
  Результаты сохранены в synthetic_datasets/thoughts_anthropic_claude-haiku-4.5.xlsx


# Linker

In [3]:
# после тестирования атомайзера выявлено, что лучшая модель атомайзера - gemini-2.5-flash
# на ее результатах будем прогонять линкер и генрировать эталонные данные для линкера (эталонные связи)

## reference data

In [6]:
best_atomizer_result = "synthetic_datasets/atomizer/houghts_google_gemini-2.5-flash.xlsx"
OUPUT_PATH = "synthetic_datasets/linker"

generate_llm_model = 'openai/gpt-5.2'
llm_models

['openai/gpt-4o-mini', 'google/gemini-2.5-flash', 'anthropic/claude-haiku-4.5']

In [5]:
# данные после лучшей модеои атомайзера 
raw_cards = pd.read_excel('synthetic_datasets/atomizer/thoughts_google_gemini-2.5-flash.xlsx')
# выбираем 30 заметок для текста
np.random.seed(48)   # для воспроизводимости

sample_note_ids = (
    raw_cards["note_id"]
    .drop_duplicates()
    .sample(n=30, random_state=42)
)
raw_cards = raw_cards[
    raw_cards["note_id"].isin(sample_note_ids)
]
raw_cards = raw_cards.sort_values(
    ["created_at", "note_id", "luhmann_id"]
).reset_index(drop=True)
raw_cards.shape, raw_cards['note_id'].nunique()

raw_cards.to_excel('synthetic_datasets/atomizer/thoughts_google_gemini-2.5-flash_500_samples.xlsx')
raw_cards.head(3)

,model_name,note_id,note_text,zettel_id,luhmann_id,parent_id,parent_luhmann_id,content,thought_type,tags,parent_hint,is_root_topic,created_at,embedding,latency_ms_total,latency_ms_per_card,cards_count
0,google/gemini-2.5-flash,e942a731-c155-4149-9b33-e4cd7540b695,Только что был звонок с Мариной из SkyLearn и ...,661a1dd0-9b96-4271-b61e-96f27ac25fd3,1,NaN,NaN,Состоялся звонок с Мариной из SkyLearn и Томом...,context,"марина, skylearn, том_harris, проект_сова, кор...",NaN,True,2026-06-27 23:51:59.187,NaN,8593.22,477.4,18
1,google/gemini-2.5-flash,e942a731-c155-4149-9b33-e4cd7540b695,Только что был звонок с Мариной из SkyLearn и ...,dc86995d-7544-48ea-a025-10df6c4998e5,1.1,661a1dd0-9b96-4271-b61e-96f27ac25fd3,1,Марина из SkyLearn и Том Harris подтвердили же...,fact,"марина, том_harris, проект_сова, 120_менеджеро...",Состоялся звонок с Мариной из SkyLearn и Томом...,False,2026-06-27 23:51:59.187,NaN,8593.22,477.4,18
2,google/gemini-2.5-flash,e942a731-c155-4149-9b33-e4cd7540b695,Только что был звонок с Мариной из SkyLearn и ...,1310950d-d980-4f69-8d5c-20ba9e0dd39f,1.1a,dc86995d-7544-48ea-a025-10df6c4998e5,1.1,Марина из SkyLearn и Том Harris просят предост...,action,"марина, том_harris, moodle, power_bi, пятница",Марина из SkyLearn и Том Harris подтвердили же...,False,2026-06-27 23:51:59.187,NaN,8593.22,477.4,18


In [6]:
# собирем обратно к ввиду ZettelCard
def df_to_zettel_cards(note_df: pd.DataFrame) -> List[ZettelCard]:
    """
    Конвертирует строки DataFrame в список Pydantic-схем ZettelCard.
    Автоматически обрабатывает NaN от Pandas и конвертирует строки обратно в списки.
    """
    cards = []
    
    # 1. Заменяем все NaN (float) на None, чтобы Pydantic корректно отработал Optional поля
    df_cleaned = note_df.replace({np.nan: None})
    
    for _, row in df_cleaned.iterrows():
        # Переводим строку в словарь для удобной распаковки
        row_dict = row.to_dict()
        
        # 2. Страховка для поля tags (если датафрейм читался из CSV, список мог стать строкой)
        tags = row_dict.get('tags')
        if tags is None:
            row_dict['tags'] = []
        elif isinstance(tags, str):
            try:
                # Пытаемся распарсить строку вида "['tag1', 'tag2']"
                row_dict['tags'] = ast.literal_eval(tags)
            except (ValueError, SyntaxError):
                # Если это просто строка через запятую "tag1, tag2"
                row_dict['tags'] = [t.strip() for t in tags.split(',') if t.strip()]
                
        # 3. Страховка для embedding (аналогично tags)
        embedding = row_dict.get('embedding')
        if isinstance(embedding, str):
            try:
                row_dict['embedding'] = ast.literal_eval(embedding)
            except (ValueError, SyntaxError):
                row_dict['embedding'] = None

        # Очищаем словарь от ключей, которых нет в схеме ZettelCard (чтобы избежать unexpected keyword argument)
        valid_keys = ZettelCard.__fields__.keys()
        filtered_dict = {k: v for k, v in row_dict.items() if k in valid_keys}

        # 4. Инициализируем карточку через распаковку словаря
        card = ZettelCard(**filtered_dict)
        cards.append(card)
        
    return cards

raw_cards = df_to_zettel_cards(raw_cards)
raw_cards[:5]

/var/folders/39/j65jq3l57mzdxq4wb5b784740000gn/T/ipykernel_4277/4210569354.py:37: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  valid_keys = ZettelCard.__fields__.keys()


[ZettelCard(zettel_id='661a1dd0-9b96-4271-b61e-96f27ac25fd3', luhmann_id='1', parent_id=None, parent_luhmann_id=None, content='Состоялся звонок с Мариной из SkyLearn и Томом Harris по пилотному проекту "Сова" для корпоративного обучения.', thought_type='context', tags=['марина', 'skylearn', 'том_harris', 'проект_сова', 'корпоративное_обучение'], parent_hint=None, is_root_topic=True, created_at=Timestamp('2026-06-27 23:51:59.187000'), embedding=None),
 ZettelCard(zettel_id='dc86995d-7544-48ea-a025-10df6c4998e5', luhmann_id='1.1', parent_id='661a1dd0-9b96-4271-b61e-96f27ac25fd3', parent_luhmann_id='1', content='Марина из SkyLearn и Том Harris подтвердили желание начать пилотный проект "Сова" 15 июля для группы из 120 менеджеров.', thought_type='fact', tags=['марина', 'том_harris', 'проект_сова', '120_менеджеров', '15_июля'], parent_hint='Состоялся звонок с Мариной из SkyLearn и Томом Harris по пилотному проекту "Сова" для корпоративного обучения.', is_root_topic=False, created_at=Timesta

In [ ]:
user_id = 'linker_eval_gpt-5.2' #
linker = GraphLinker(
    embedding_model=embedding_model,
    model_name=generate_llm_model,
    temperature=settings.linker_temperature,
    system_prompt=settings.linker_system_prompt,
    user_prompt_template=settings.linker_user_prompt_template,
    similarity_threshold=settings.linker_similarity_threshold,
    max_candidates=settings.linker_max_candidates,
)

start_time = time.time()
results = linker.link_and_insert(user_id=user_id, new_cards=raw_cards)
end_time = time.time()
print(f"Время выполнения: {end_time - start_time} секунд")
results

In [55]:
ref_list = []

for link in results:
    # Безопасно извлекаем атрибуты из объекта ZettelNode (link.card)
    card_node = link.card
    
    ref_list.append({
        # 1. Основные метаданные карточки (ZettelNode)
        'zettel_id': getattr(card_node, 'zettel_id', None),
        'luhmann_id': getattr(card_node, 'luhmann_id', None),
        'content': getattr(card_node, 'content', None),
        'thought_type': getattr(card_node, 'thought_type', None),
        'tags': getattr(card_node, 'tags', []),
        'is_root_topic': getattr(card_node, 'is_root_topic', False),
        
        # Дополнительные поля (системные), если они пишутся в ZettelNode при создании
        'user_id': getattr(card_node, 'user_id', None),
        'embedding': getattr(card_node, 'embedding', None),
        'created_at': getattr(card_node, 'created_at', None),
        'updated_at': getattr(card_node, 'updated_at', None),
        'similarity': getattr(card_node, 'similarity', None),
        
        # 2. Результаты решения Linker (LinkResult)
        'action': link.action.value if link.action else None, # Извлекаем строковое значение из Enum
        'reasoning': link.reasoning,
        'candidates_found': link.candidates_found,
        'target_zettel_id': link.target_zettel_id,
    })

# Создаем датафрейм
df_linker_ref = pd.DataFrame(ref_list)
print(f"Собрано строк: {len(df_linker_ref)}")
df_linker_ref.to_excel('synthetic_datasets/linker/reference_linker_gpt-5.2.xlsx')
df_linker_ref.head()

Собрано строк: 484


,zettel_id,luhmann_id,content,thought_type,tags,is_root_topic,user_id,embedding,created_at,updated_at,similarity,action,reasoning,candidates_found,target_zettel_id
0,c6bf16a2-c1b0-4a0b-a61e-8e5db768de7a,1,Состоялся звонок с Мариной из SkyLearn и Томом...,context,"[марина, skylearn, том_harris, проект_сова, ко...",True,linker_eval_gpt-5.2,"[-0.006099278572946787, 0.08536402881145477, -...",None,None,None,new_root,Нет похожих карточек в графе,0,NaN
1,246d96fc-0403-479e-a421-c0eed015b7d3,1.1,Марина из SkyLearn и Том Harris подтвердили же...,fact,"[марина, том_harris, проект_сова, 120_менеджер...",False,linker_eval_gpt-5.2,"[0.00489732064306736, 0.07277049869298935, -0....",None,None,None,child_of,Дочерняя карточка внутри текущего сообщения.,0,NaN
2,4ca48a62-c6c2-4a31-9083-2f7a0b9c54b4,1.1a,Марина из SkyLearn и Том Harris просят предост...,action,"[марина, том_harris, moodle, power_bi, пятница]",False,linker_eval_gpt-5.2,"[0.008820367977023125, 0.06661751121282578, -0...",None,None,None,child_of,Дочерняя карточка внутри текущего сообщения.,0,NaN
3,418cc6c1-a3a0-4f97-a694-ea26b2253aa2,1.1b,"Марина из SkyLearn сообщила, что текущий подря...",fact,"[марина, skylearn, eduspace, кабинет_наставника]",False,linker_eval_gpt-5.2,"[-0.009722822345793247, 0.06320757418870926, -...",None,None,None,child_of,Дочерняя карточка внутри текущего сообщения.,0,NaN
4,8fd300d0-bb61-4d5f-89f6-77c0870356dc,1.1b1,NPS у SkyLearn просел с 62 до 41 из-за срыва р...,fact,"[nps, skylearn, eduspace, кабинет_наставника]",False,linker_eval_gpt-5.2,"[-0.013034241273999214, 0.04073932021856308, -...",None,None,None,child_of,Дочерняя карточка внутри текущего сообщения.,0,NaN


'new_root'

## test models

прогоняем на тестовых моделях

In [58]:
llm_models

['openai/gpt-4o-mini', 'google/gemini-2.5-flash', 'anthropic/claude-haiku-4.5']

In [ ]:
# Опционально: список для сбора всех датафреймов, если потом захочешь склеить их в один
all_models_results = [] 

for model in tqdm(llm_models, desc="Evaluating LLM Models"):
    print(f'        LLM model: {model}')
    # Заменяем запрещенные для файловой системы символы (например, / или :)
    safe_model_name = str(model).replace('/', '_').replace(':', '_')
    
    user_id = f'linker_eval_{safe_model_name}'
    
    linker = GraphLinker(
        embedding_model=embedding_model,
        model_name=model,
        temperature=settings.linker_temperature,
        system_prompt=settings.linker_system_prompt,
        user_prompt_template=settings.linker_user_prompt_template,
        similarity_threshold=settings.linker_similarity_threshold,
        max_candidates=settings.linker_max_candidates,
    )
    
    start_time = time.time()
    # ИСПРАВЛЕНО: передаем динамический user_id, а не 'demo_user_001'
    results = linker.link_and_insert(user_id=user_id, new_cards=raw_cards)
    end_time = time.time()
    
    # Считаем общее время на прогон текущей модели
    latency_sec = end_time - start_time

    df_list = []
    
    for link in results:
        card_node = link.card
        
        df_list.append({
            'zettel_id': getattr(card_node, 'zettel_id', None),
            'luhmann_id': getattr(card_node, 'luhmann_id', None),
            'content': getattr(card_node, 'content', None),
            'thought_type': getattr(card_node, 'thought_type', None),
            'tags': getattr(card_node, 'tags', []),
            'is_root_topic': getattr(card_node, 'is_root_topic', False),
            'user_id': getattr(card_node, 'user_id', None),
            'embedding': getattr(card_node, 'embedding', None),
            'created_at': getattr(card_node, 'created_at', None),
            'updated_at': getattr(card_node, 'updated_at', None),
            'similarity': getattr(card_node, 'similarity', None),
            
            'action': link.action.value if link.action else None,
            'reasoning': link.reasoning,
            'candidates_found': link.candidates_found,
            'target_zettel_id': link.target_zettel_id,
            
            # ИСПРАВЛЕНО: добавлено значение для latency и колонка с именем модели
            'latency_sec': latency_sec,
            'model_name': model
        })

    # Создаем датафрейм для текущей модели
    df_linker= pd.DataFrame(df_list)
    
    print(f"\n  -->Модель: {model} | Собрано строк: {len(df_linker)} | Время: {latency_sec:.2f} сек")
    
    # ИСПРАВЛЕНО: сохраняем файл с уникальным именем для каждой модели
    file_path = f'synthetic_datasets/linker/linker_{safe_model_name}.xlsx'
    df_linker.to_excel(file_path, index=False)
    
    all_models_results.append(df_linker)

# Выводим первые 5 строк последней отработавшей модели для визуальной проверки
df_linker.head()